## tl;dr

Across **490** valid paired states from **100** current non-candle binary markets and **5** public batch rounds, the registered residual rule rejected **0** states. The stricter fixed 0.002 robustness threshold rejected **0**. This independently supports zero observed residual-magnitude selectivity after paired-top validity, but it is label-free and cannot establish settlement prediction or profitability. Pair timestamp skew reached 2 ms and is retained as a limitation; the active frozen rule remains unchanged.


## Context & Methods

This is an **independent, label-free mechanism replication** for the frozen
`binary_complement_coherence_v1` rule. It asks whether the registered complement-residual
magnitude rejects any valid paired book outside the BTC candle universe.

The sample is the deterministic top 100 currently active, accepting, non-crypto,
non-negative-risk binary Polymarket markets by 24-hour volume from five paginated Gamma
pages. Five public `POST /books` batches captured both outcome tokens for every market.
No resolution labels, strategy outcomes, BTC reference tapes, or alternate residual
threshold search are used. This can test observed structural selectivity; it cannot test
settlement prediction, PnL, or causal profitability.

### Key Assumptions

- Each selected market has exactly the ordered outcomes `[Yes, No]` and two corresponding token IDs.
- Best bid/ask are derived by price extrema rather than array position because the observed REST arrays conflict with the documented sort direction.
- The registered comparison mirrors the Rust live path: start from the maximum pair-declared tick, then causally tighten it to 0.001 when either paired top is outside `[0.04, 0.96]`. A fixed 0.002 threshold is reported only as a stricter robustness check for states whose effective tick remains 0.01, not a selected replacement.
- Batch responses are not assumed atomic; pair timestamp skew is measured explicitly.

## Data

In [1]:
from __future__ import annotations

import gzip
import hashlib
import json
import math
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path('/Users/ttoomm/Documents/PolyMomentum')
SNAPSHOT_PATH = ROOT / 'deploy/promotions/evidence/strategy_registry/source_snapshots/20260721_non_candle_public_books.json.gz'
MANIFEST_PATH = ROOT / 'deploy/promotions/evidence/strategy_registry/source_snapshots/20260721_non_candle_public_books_manifest.json'
OUTPUT_PATH = ROOT / 'deploy/promotions/evidence/strategy_registry/20260721_binary_complement_residual_cross_market_replication.json'
DEPTH_LEVELS = 3
EPSILON = 1e-12

def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()

def distribution(values: list[float]) -> dict[str, float | int | None]:
    finite = sorted(float(value) for value in values if math.isfinite(float(value)))
    if not finite:
        return {'count': 0, 'min': None, 'p50': None, 'p90': None, 'p99': None, 'max': None, 'mean': None}
    def quantile(probability: float) -> float:
        index = probability * (len(finite) - 1)
        lower = int(math.floor(index))
        upper = int(math.ceil(index))
        if lower == upper:
            return finite[lower]
        weight = index - lower
        return finite[lower] * (1.0 - weight) + finite[upper] * weight
    return {
        'count': len(finite), 'min': finite[0], 'p50': quantile(0.50),
        'p90': quantile(0.90), 'p99': quantile(0.99), 'max': finite[-1],
        'mean': sum(finite) / len(finite),
    }

def pearson(xs: list[float], ys: list[float]) -> float | None:
    if len(xs) != len(ys) or len(xs) < 2:
        return None
    mean_x = sum(xs) / len(xs)
    mean_y = sum(ys) / len(ys)
    numerator = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys))
    denominator_x = sum((x - mean_x) ** 2 for x in xs)
    denominator_y = sum((y - mean_y) ** 2 for y in ys)
    if denominator_x <= EPSILON or denominator_y <= EPSILON:
        return None
    return numerator / math.sqrt(denominator_x * denominator_y)

manifest = json.loads(MANIFEST_PATH.read_text())
compressed = SNAPSHOT_PATH.read_bytes()
canonical = gzip.decompress(compressed)
assert sha256_bytes(compressed) == manifest['snapshot_sha256']
assert sha256_bytes(canonical) == manifest['snapshot_uncompressed_sha256']
snapshot = json.loads(canonical)
assert snapshot['schema_version'] == 1
assert len(snapshot['markets']) == 100
assert len(snapshot['rounds']) == 5
assert all(len(round_data['books']) == 200 for round_data in snapshot['rounds'])
assert snapshot['selection']['selected_tokens'] == 200

print({
    'captured_at': snapshot['captured_at'],
    'markets': len(snapshot['markets']),
    'rounds': len(snapshot['rounds']),
    'books': sum(len(round_data['books']) for round_data in snapshot['rounds']),
    'snapshot_sha256': manifest['snapshot_sha256'],
})

{'captured_at': '2026-07-21T06:34:38.487694+00:00', 'markets': 100, 'rounds': 5, 'books': 1000, 'snapshot_sha256': '5fd25a3e78e1f46c556b1adcb6e05d0939f5fd475d4ddf48136863588fc78811'}


### Reconstruct valid paired top-of-book states

In [2]:
def price_levels(raw_levels: list[dict], reverse: bool) -> list[tuple[float, float]]:
    levels = [
        (float(level['price']), float(level['size']))
        for level in raw_levels
        if float(level['size']) > 0
    ]
    return sorted(levels, reverse=reverse)

def book_metrics(book: dict) -> tuple[dict | None, str | None]:
    bids = price_levels(book['bids'], reverse=True)
    asks = price_levels(book['asks'], reverse=False)
    if not bids or not asks:
        return None, 'missing_side'
    best_bid = bids[0][0]
    best_ask = asks[0][0]
    if not (0.0 < best_bid < best_ask < 1.0):
        return None, 'invalid_top'
    bid_depth = sum(size for _, size in bids[:DEPTH_LEVELS])
    ask_depth = sum(size for _, size in asks[:DEPTH_LEVELS])
    if bid_depth <= 0 or ask_depth <= 0:
        return None, 'missing_positive_depth'
    midpoint = (best_bid + best_ask) / 2.0
    microprice = (best_ask * bid_depth + best_bid * ask_depth) / (bid_depth + ask_depth)
    return {
        'best_bid': best_bid,
        'best_ask': best_ask,
        'bid_depth': bid_depth,
        'ask_depth': ask_depth,
        'midpoint': midpoint,
        'microprice': microprice,
        'spread': best_ask - best_bid,
        'timestamp_ms': int(book['timestamp']),
        'tick_size': float(book['tick_size']),
        'market': str(book['market']),
        'neg_risk': bool(book['neg_risk']),
    }, None

market_by_condition = {market['condition_id']: market for market in snapshot['markets']}
expected_tokens = {
    token_id
    for market in snapshot['markets']
    for token_id in market['token_ids']
}
rows = []
invalid_reasons = Counter()
documented_sort_checks = Counter()
condition_id_mismatches = 0
negative_risk_books = 0
token_coverage_failures = 0

for round_data in snapshot['rounds']:
    books = {str(book['asset_id']): book for book in round_data['books']}
    if set(books) != expected_tokens or len(books) != len(expected_tokens):
        token_coverage_failures += 1
    for book in round_data['books']:
        bid_prices = [float(level['price']) for level in book['bids']]
        ask_prices = [float(level['price']) for level in book['asks']]
        documented_sort_checks['books'] += 1
        documented_sort_checks['bids_descending'] += int(
            all(left >= right for left, right in zip(bid_prices, bid_prices[1:]))
        )
        documented_sort_checks['asks_ascending'] += int(
            all(left <= right for left, right in zip(ask_prices, ask_prices[1:]))
        )
    for condition_id, market in market_by_condition.items():
        yes_token, no_token = market['token_ids']
        yes_book = books.get(yes_token)
        no_book = books.get(no_token)
        base = {'round_index': round_data['round_index'], 'condition_id': condition_id}
        if yes_book is None or no_book is None:
            invalid_reasons['missing_book'] += 1
            rows.append({**base, 'valid': False, 'invalid_reason': 'missing_book'})
            continue
        yes, yes_reason = book_metrics(yes_book)
        no, no_reason = book_metrics(no_book)
        if yes is None or no is None:
            reason = yes_reason or no_reason or 'unknown'
            invalid_reasons[reason] += 1
            rows.append({**base, 'valid': False, 'invalid_reason': reason})
            continue
        condition_id_mismatches += int(yes['market'] != condition_id or no['market'] != condition_id)
        negative_risk_books += int(yes['neg_risk']) + int(no['neg_risk'])
        midpoint_residual = yes['midpoint'] + no['midpoint'] - 1.0
        microprice_residual = yes['microprice'] + no['microprice'] - 1.0
        declared_tick = max(yes['tick_size'], no['tick_size'])
        current_band_tick = 0.001 if (
            min(yes['best_bid'], yes['best_ask'], no['best_bid'], no['best_ask']) < 0.04
            or max(yes['best_bid'], yes['best_ask'], no['best_bid'], no['best_ask']) > 0.96
        ) else 0.01
        effective_live_tick = min(declared_tick, current_band_tick)
        threshold = 2.0 * effective_live_tick
        max_abs_residual = max(abs(midpoint_residual), abs(microprice_residual))
        rows.append({
            **base,
            'valid': True,
            'midpoint_residual': midpoint_residual,
            'microprice_residual': microprice_residual,
            'max_abs_residual': max_abs_residual,
            'declared_tick': declared_tick,
            'current_band_tick': current_band_tick,
            'effective_live_tick': effective_live_tick,
            'threshold': threshold,
            'fixed_pass': max_abs_residual <= threshold + EPSILON,
            'strict_0_002_pass': max_abs_residual <= 0.002 + EPSILON,
            'timestamp_skew_ms': abs(yes['timestamp_ms'] - no['timestamp_ms']),
            'spread_sum': yes['spread'] + no['spread'],
            'cross_touch_abs': max(
                abs(yes['best_bid'] + no['best_ask'] - 1.0),
                abs(yes['best_ask'] + no['best_bid'] - 1.0),
            ),
            'depth_mirror_abs': max(
                abs(yes['bid_depth'] - no['ask_depth']),
                abs(yes['ask_depth'] - no['bid_depth']),
            ),
            'same_book_hash': yes_book['hash'] == no_book['hash'],
        })

valid_rows = [row for row in rows if row['valid']]
assert len(rows) == 500
assert token_coverage_failures == 0
assert condition_id_mismatches == 0
assert negative_risk_books == 0

data_quality = {
    'possible_paired_states': len(rows),
    'valid_paired_states': len(valid_rows),
    'invalid_reason_counts': dict(invalid_reasons),
    'token_coverage_failures': token_coverage_failures,
    'condition_id_mismatches': condition_id_mismatches,
    'negative_risk_books': negative_risk_books,
    'documented_sort_direction_checks': dict(documented_sort_checks),
    'best_price_policy': 'price extrema; do not trust REST array position',
    'pair_timestamp_skew_ms': distribution([row['timestamp_skew_ms'] for row in valid_rows]),
    'states_dynamic_transition_tightens_declared_tick': sum(
        row['effective_live_tick'] + EPSILON < row['declared_tick'] for row in valid_rows
    ),
    'states_declared_tick_already_dynamic_without_current_band_cross': sum(
        row['declared_tick'] <= 0.001 + EPSILON and row['current_band_tick'] > 0.001 + EPSILON
        for row in valid_rows
    ),
    'unique_round_response_hashes': len({round_data['raw_response_sha256'] for round_data in snapshot['rounds']}),
}
print(data_quality)

{'possible_paired_states': 500, 'valid_paired_states': 490, 'invalid_reason_counts': {'missing_side': 10}, 'token_coverage_failures': 0, 'condition_id_mismatches': 0, 'negative_risk_books': 0, 'documented_sort_direction_checks': {'books': 1000, 'bids_descending': 10, 'asks_ascending': 10}, 'best_price_policy': 'price extrema; do not trust REST array position', 'pair_timestamp_skew_ms': {'count': 490, 'min': 0.0, 'p50': 0.0, 'p90': 0.0, 'p99': 1.0, 'max': 2.0, 'mean': 0.024489795918367346}, 'states_dynamic_transition_tightens_declared_tick': 0, 'states_declared_tick_already_dynamic_without_current_band_cross': 97, 'unique_round_response_hashes': 5}


## Results

### The registered residual magnitude again contributes no observed rejection

In [3]:
fixed_rejections = [row for row in valid_rows if not row['fixed_pass']]
strict_rejections = [row for row in valid_rows if not row['strict_0_002_pass']]
midpoint_rejections = [
    row for row in valid_rows
    if abs(row['midpoint_residual']) > row['threshold'] + EPSILON
]
microprice_incremental_rejections = [
    row for row in valid_rows
    if abs(row['midpoint_residual']) <= row['threshold'] + EPSILON
    and abs(row['microprice_residual']) > row['threshold'] + EPSILON
]

per_round = []
for round_index in range(5):
    round_rows = [row for row in valid_rows if row['round_index'] == round_index]
    per_round.append({
        'round_index': round_index,
        'valid_pairs': len(round_rows),
        'fixed_rejections': sum(not row['fixed_pass'] for row in round_rows),
        'max_abs_residual': max((row['max_abs_residual'] for row in round_rows), default=None),
        'timestamp_skew_ms_max': max((row['timestamp_skew_ms'] for row in round_rows), default=None),
    })

per_condition = []
for condition_id, market in market_by_condition.items():
    condition_rows = [row for row in valid_rows if row['condition_id'] == condition_id]
    per_condition.append({
        'condition_id': condition_id,
        'question': market['question'],
        'valid_rounds': len(condition_rows),
        'fixed_rejections': sum(not row['fixed_pass'] for row in condition_rows),
        'max_abs_residual': max((row['max_abs_residual'] for row in condition_rows), default=None),
    })

structural_results = {
    'possible_paired_states': len(rows),
    'valid_paired_states': len(valid_rows),
    'valid_pair_coverage': len(valid_rows) / len(rows),
    'registered_fixed_rule_passes': len(valid_rows) - len(fixed_rejections),
    'registered_fixed_rule_rejections': len(fixed_rejections),
    'conditions_with_registered_rejection': sum(row['fixed_rejections'] > 0 for row in per_condition),
    'midpoint_clause_rejections': len(midpoint_rejections),
    'microprice_incremental_rejections_beyond_midpoint': len(microprice_incremental_rejections),
    'strict_0_002_threshold_rejections': len(strict_rejections),
    'max_abs_residual': distribution([row['max_abs_residual'] for row in valid_rows]),
    'maximum_registered_threshold_utilization': max(
        (row['max_abs_residual'] / row['threshold'] for row in valid_rows), default=None
    ),
    'minimum_registered_threshold_headroom': min(
        (row['threshold'] - row['max_abs_residual'] for row in valid_rows), default=None
    ),
    'midpoint_residual_nonzero_states': sum(abs(row['midpoint_residual']) > EPSILON for row in valid_rows),
    'microprice_residual_nonzero_states': sum(abs(row['microprice_residual']) > EPSILON for row in valid_rows),
    'cross_touch_error_nonzero_states': sum(row['cross_touch_abs'] > EPSILON for row in valid_rows),
    'depth_mirror_mismatch_states': sum(row['depth_mirror_abs'] > EPSILON for row in valid_rows),
    'same_pair_book_hash_states': sum(row['same_book_hash'] for row in valid_rows),
    'per_round': per_round,
    'per_condition': sorted(
        per_condition,
        key=lambda row: (
            row['max_abs_residual'] is None,
            -(row['max_abs_residual'] if row['max_abs_residual'] is not None else 0.0),
            row['condition_id'],
        ),
    ),
}

print({key: value for key, value in structural_results.items() if key not in {'per_condition'}})
print({'largest_residual_markets': structural_results['per_condition'][:10]})

{'possible_paired_states': 500, 'valid_paired_states': 490, 'valid_pair_coverage': 0.98, 'registered_fixed_rule_passes': 490, 'registered_fixed_rule_rejections': 0, 'conditions_with_registered_rejection': 0, 'midpoint_clause_rejections': 0, 'microprice_incremental_rejections_beyond_midpoint': 0, 'strict_0_002_threshold_rejections': 0, 'max_abs_residual': {'count': 490, 'min': 0.0, 'p50': 0.0, 'p90': 2.220446049250313e-16, 'p99': 1.3739489307216733e-05, 'max': 0.0001179054732232121, 'mean': 1.2066478568532794e-06}, 'maximum_registered_threshold_utilization': 0.05895273661160605, 'minimum_registered_threshold_headroom': 0.001882094526776788, 'midpoint_residual_nonzero_states': 0, 'microprice_residual_nonzero_states': 7, 'cross_touch_error_nonzero_states': 0, 'depth_mirror_mismatch_states': 7, 'same_pair_book_hash_states': 0, 'per_round': [{'round_index': 0, 'valid_pairs': 98, 'fixed_rejections': 0, 'max_abs_residual': 0.0001179054732232121, 'timestamp_skew_ms_max': 2}, {'round_index': 1,

### Residual exceptions track depth races, not spread or timestamp skew

In [4]:
abs_micro = [abs(row['microprice_residual']) for row in valid_rows]
driver_result = {
    'correlations_with_abs_microprice_residual': {
        field: pearson(abs_micro, [float(row[field]) for row in valid_rows])
        for field in ('timestamp_skew_ms', 'spread_sum', 'depth_mirror_abs')
    },
    'nonzero_microprice_and_depth_mismatch_overlap': sum(
        abs(row['microprice_residual']) > EPSILON and row['depth_mirror_abs'] > EPSILON
        for row in valid_rows
    ),
    'nonzero_microprice_and_timestamp_skew_overlap': sum(
        abs(row['microprice_residual']) > EPSILON and row['timestamp_skew_ms'] > 0
        for row in valid_rows
    ),
    'timestamp_skew_zero_states': sum(row['timestamp_skew_ms'] == 0 for row in valid_rows),
    'max_depth_mirror_mismatch': max((row['depth_mirror_abs'] for row in valid_rows), default=None),
}
print(driver_result)

{'correlations_with_abs_microprice_residual': {'timestamp_skew_ms': 0.5241602643088302, 'spread_sum': -0.06071974152408427, 'depth_mirror_abs': 0.9624010647810752}, 'nonzero_microprice_and_depth_mismatch_overlap': 7, 'nonzero_microprice_and_timestamp_skew_overlap': 7, 'timestamp_skew_zero_states': 481, 'max_depth_mirror_mismatch': 81.85999999999996}


### Persist the source-pinned replication artifact

In [5]:
all_zero_selectivity = structural_results['registered_fixed_rule_rejections'] == 0
status = (
    'DIAGNOSTIC_ONLY_CROSS_MARKET_REPLICATION_ZERO_REGISTERED_RESIDUAL_REJECTIONS_ACTIVE_RULE_UNCHANGED'
    if all_zero_selectivity
    else 'DIAGNOSTIC_ONLY_CROSS_MARKET_REPLICATION_OBSERVED_REGISTERED_RESIDUAL_REJECTIONS_ACTIVE_RULE_UNCHANGED'
)

evidence = {
    'schema_version': 1,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'mechanism_id': 'binary_complement_coherence_v1',
    'status': status,
    'decision_question': 'Does the registered two-live-tick residual magnitude add observed selectivity beyond valid paired books in an independent current non-candle market sample?',
    'source_authority': {
        'snapshot_path': str(SNAPSHOT_PATH.relative_to(ROOT)),
        'manifest_path': str(MANIFEST_PATH.relative_to(ROOT)),
        'snapshot_sha256': manifest['snapshot_sha256'],
        'snapshot_uncompressed_sha256': manifest['snapshot_uncompressed_sha256'],
        'captured_at': snapshot['captured_at'],
        'gamma_rows_loaded': snapshot['selection']['gamma_rows_loaded'],
        'eligible_before_rank_limit': snapshot['selection']['eligible_before_rank_limit'],
        'selected_markets': len(snapshot['markets']),
        'capture_rounds': len(snapshot['rounds']),
        'official_docs': snapshot['source']['official_docs'],
        'resolution_labels_loaded': False,
        'strategy_outcomes_loaded': False,
        'btc_reference_tapes_loaded': False,
    },
    'population': {
        'definition': 'top 100 active, accepting, non-crypto, non-negative-risk binary CLOB markets by volume24hr after deterministic filtering of the top 500 Gamma rows',
        'selection_order': snapshot['selection']['selection_order'],
        'excluded_text_regex': snapshot['selection']['excluded_text_regex'],
        'markets': len(snapshot['markets']),
        'rounds': len(snapshot['rounds']),
        'possible_paired_states': len(rows),
    },
    'methodology': {
        'book_depth_levels': DEPTH_LEVELS,
        'best_price_policy': data_quality['best_price_policy'],
        'registered_comparison': 'max(abs(midpoint residual), abs(microprice residual)) <= 2 * min(max(pair-declared ticks), causal current-band tick)',
        'strict_robustness_threshold': 0.002,
        'alternate_threshold_search_or_selection': False,
        'batch_atomicity_assumed': False,
        'pair_timestamp_skew_measured': True,
    },
    'data_quality': {
        **data_quality,
        'quality_assessment': 'READY_WITH_CAVEATS_FOR_LABEL_FREE_CROSS_MARKET_MECHANISM_REPLICATION_ONLY',
        'documented_sort_conflict': 'Observed bid/ask arrays did not follow the current documented best-first direction; price extrema were used and array indices were not trusted.',
        'promotion_or_profitability_eligible': False,
    },
    'structural_results': structural_results,
    'driver_diagnostic': driver_result,
    'mechanism_assessment': {
        'registered_residual_magnitude_selectivity': (
            'ZERO_OBSERVED_REJECTIONS_IN_INDEPENDENT_CURRENT_SAMPLE'
            if all_zero_selectivity else 'OBSERVED_REJECTIONS_REQUIRE_REVIEW'
        ),
        'paired_top_validity': 'REQUIRED_PRECONDITION_NOT_AN_OUTCOME_SIGNAL',
        'interpretation': (
            'The current cross-market replication supports the earlier finding that the registered residual magnitude is observationally inert once both binary outcome books have valid tops.'
            if all_zero_selectivity
            else 'The independent sample contains residual rejections and does not replicate zero selectivity.'
        ),
        'active_binary_complement_rule_changed': False,
        'why_not_change_now': 'The forward BTC block is frozen and sealed; this cross-market sample is label-free and cannot measure economic value.',
    },
    'decision': {
        'strategy_adjustment': 'REGISTER_POST_BLOCK_PAIR_VALIDITY_ABLATION; NO_ACTIVE_RULE_OR_PARAMETER_CHANGE',
        'research_action': 'After the sealed block decision, compare the frozen rule with a paired-top-validity-only ablation on the same disjoint labeled evidence before crediting or retaining residual magnitude.',
        'a_plus_claim': False,
        'profitability_claim': False,
        'live_trading': 'OFF',
    },
    'limitations': [
        'This is a five-round snapshot over roughly nine seconds, not a full market lifecycle.',
        'The 100-market sample is volume-ranked and intentionally excludes crypto/candle and negative-risk markets; it is independent but not representative of every Polymarket market.',
        'Batch retrieval is not guaranteed atomic; observed per-book timestamp skew is measured but cannot reconstruct a common exchange instant.',
        'The stricter 0.002 threshold is a robustness check only and is not a searched or proposed live parameter.',
        'No terminal labels or execution simulation are loaded, so the artifact cannot measure prediction, losses removed, PnL, or profitability.',
    ],
}

temporary = OUTPUT_PATH.with_name(f'{OUTPUT_PATH.name}.tmp')
temporary.write_text(json.dumps(evidence, indent=2, sort_keys=True) + '\n')
temporary.replace(OUTPUT_PATH)
print({'artifact': str(OUTPUT_PATH.relative_to(ROOT)), 'status': status})

{'artifact': 'deploy/promotions/evidence/strategy_registry/20260721_binary_complement_residual_cross_market_replication.json', 'status': 'DIAGNOSTIC_ONLY_CROSS_MARKET_REPLICATION_ZERO_REGISTERED_RESIDUAL_REJECTIONS_ACTIVE_RULE_UNCHANGED'}


## Takeaways

- Registered residual rejections: **0 / 490** valid paired states.
- Midpoint rejections: **0**; incremental microprice rejections: **0**.
- Even the fixed 0.002 robustness threshold rejected **0** states.
- The evidence registers a post-block paired-top-validity-only ablation; it does not change the sealed rule and makes no A+ or profitability claim.
